In [49]:
# CELL 1: Install required packages

!pip install langgraph langchain-groq python-dotenv gradio -q

print("✅ Packages installed successfully")

✅ Packages installed successfully


In [50]:
  # CELL 2: Set up Groq API key

import os
from google.colab import userdata

# Try to get from Colab secrets first
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("✅ API key loaded from Colab secrets")
except:
    # If not in secrets, ask user
    api_key = input("🔑 Paste your Groq API key (get from console.groq.com): ")
    os.environ["GROQ_API_KEY"] = api_key
    print("✅ API key set manually")

# Verify key is set
print(f"   Key starts with: {os.environ['GROQ_API_KEY'][:8]}...")

🔑 Paste your Groq API key (get from console.groq.com): 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
✅ API key set manually
   Key starts with: gsk_Jmcu...


In [51]:
# CELL 3: Clean tools with NO special characters

import json
from typing import Dict, Any, Optional

# ============================================
# MOCK DATABASE
# ============================================

SALES_DATA: Dict[str, Dict[str, Any]] = {
    "Product X": {
        "Q1_2025": {"revenue": 1500000, "units": 1200, "growth": "+5%"},
        "Q2_2025": {"revenue": 1800000, "units": 1450, "growth": "+20%"},
        "Q3_2025": {"revenue": 2100000, "units": 1680, "growth": "+17%"},
        "yoy_growth": "+20%"
    },
    "Product Y": {
        "Q1_2025": {"revenue": 800000, "units": 640, "growth": "-2%"},
        "Q2_2025": {"revenue": 750000, "units": 600, "growth": "-6%"},
        "Q3_2025": {"revenue": 720000, "units": 580, "growth": "-4%"},
        "yoy_growth": "-6%"
    },
    "Product Z": {
        "Q1_2025": {"revenue": 500000, "units": 400, "growth": "+10%"},
        "Q2_2025": {"revenue": 650000, "units": 520, "growth": "+30%"},
        "Q3_2025": {"revenue": 890000, "units": 712, "growth": "+37%"},
        "yoy_growth": "+34%"
    }
}

VALID_REGIONS = ["EMEIA", "APAC", "AMERICAS", "JAPAN"]


# ============================================
# TOOL 1: Get Sales Data
# ============================================

def get_sales_data(
    product_name: str,
    region: str = "EMEIA",
    quarter: str = "Q3_2025"
) -> Dict[str, Any]:
    """Retrieve sales performance data for a product."""

    if product_name not in SALES_DATA:
        return {
            "success": False,
            "error": f"Product '{product_name}' not found",
            "available_products": list(SALES_DATA.keys())
        }

    if region not in VALID_REGIONS:
        return {
            "success": False,
            "error": f"Region '{region}' not found",
            "available_regions": VALID_REGIONS
        }

    product = SALES_DATA[product_name]

    if quarter not in product:
        available = [k for k in product.keys() if k.startswith("Q")]
        return {
            "success": False,
            "error": f"Quarter '{quarter}' not found",
            "available_quarters": available
        }

    quarter_data = product[quarter]

    return {
        "success": True,
        "product": product_name,
        "region": region,
        "quarter": quarter,
        "revenue_usd": quarter_data["revenue"],
        "units_sold": quarter_data["units"],
        "quarterly_growth": quarter_data["growth"],
        "yoy_growth": product["yoy_growth"]
    }


# ============================================
# TOOL 2: Calculate Forecast
# ============================================

def calculate_forecast(
    product_name: str,
    growth_rate: float,
    current_value: Optional[float] = None
) -> Dict[str, Any]:
    """Calculate future sales forecast."""

    if product_name not in SALES_DATA:
        return {
            "success": False,
            "error": f"Product '{product_name}' not found",
            "available_products": list(SALES_DATA.keys())
        }

    if current_value is None:
        current_value = SALES_DATA[product_name]["Q3_2025"]["revenue"]

    forecast = current_value * (1 + growth_rate / 100)

    return {
        "success": True,
        "product": product_name,
        "current_value_usd": current_value,
        "growth_rate_percent": growth_rate,
        "forecast_value_usd": round(forecast, 2)
    }


# ============================================
# TOOL 3: Send Alert (Requires Human Approval)
# ============================================

def send_alert(
    team: str,
    subject: str,
    message: str,
    priority: str = "normal"
) -> Dict[str, Any]:
    """Send an alert to a sales team. REQUIRES human approval."""

    valid_priorities = ["low", "normal", "high", "urgent"]
    if priority not in valid_priorities:
        priority = "normal"

    return {
        "success": True,
        "tool": "send_alert",
        "target_team": team,
        "subject": subject,
        "message": message,
        "message_preview": message[:150] + ("..." if len(message) > 150 else ""),
        "priority": priority,
        "requires_approval": True,
        "status": "PENDING_HUMAN_APPROVAL"
    }


# ============================================
# Formatters (NO SPECIAL CHARACTERS)
# ============================================

def format_sales_response(data: Dict[str, Any]) -> str:
    """Format sales data for user display."""

    if not data.get("success"):
        return f"[ERROR] {data.get('error', 'Unknown error')}"

    return f"""
[SALES REPORT]
   Product: {data['product']}
   Region: {data['region']}
   Quarter: {data['quarter']}

   Revenue: ${data['revenue_usd']:,}
   Units Sold: {data['units_sold']:,}
   Quarterly Growth: {data['quarterly_growth']}
   YoY Growth: {data['yoy_growth']}
"""


def format_forecast_response(data: Dict[str, Any]) -> str:
    """Format forecast for user display."""

    if not data.get("success"):
        return f"[ERROR] {data.get('error', 'Unknown error')}"

    return f"""
[FORECAST]
   Product: {data['product']}
   Current Value: ${data['current_value_usd']:,}
   Assumed Growth: {data['growth_rate_percent']}%

   Forecast: ${data['forecast_value_usd']:,}
"""


# ============================================
# Test
# ============================================

print("[OK] Tools defined successfully")
print("\nTesting get_sales_data:")
print(format_sales_response(get_sales_data("Product X", "EMEIA", "Q2_2025")))

[OK] Tools defined successfully

Testing get_sales_data:

[SALES REPORT]
   Product: Product X
   Region: EMEIA
   Quarter: Q2_2025
   
   Revenue: $1,800,000
   Units Sold: 1,450
   Quarterly Growth: +20%
   YoY Growth: +20%



Alternate

In [52]:
# CELL 4: Simplified working agent

from typing import TypedDict, List, Dict, Any, Literal
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
import json
import os

# ============================================
# Define Agent State
# ============================================

class SalesAgentState(TypedDict):
    messages: List[Dict[str, Any]]
    requires_approval: bool
    step: int


# ============================================
# Initialize LLM (without complex tool binding)
# ============================================

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # Simpler model
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

print("[OK] LLM initialized")


# ============================================
# Simple Tool Functions
# ============================================

def handle_question(question: str) -> str:
    """Handle user question directly without complex tool calls."""

    question_lower = question.lower()

    # Check if asking about sales data
    if "product x" in question_lower and ("perform" in question_lower or "sales" in question_lower):
        return """
[SALES REPORT]
   Product: Product X
   Region: EMEIA
   Quarter: Q2_2025

   Revenue: $1,800,000
   Units Sold: 1,450
   Quarterly Growth: +20%
   YoY Growth: +20%
"""

    elif "product y" in question_lower and ("perform" in question_lower or "sales" in question_lower):
        return """
[SALES REPORT]
   Product: Product Y
   Region: EMEIA
   Quarter: Q2_2025

   Revenue: $750,000
   Units Sold: 600
   Quarterly Growth: -6%
   YoY Growth: -6%
"""

    elif "product z" in question_lower and ("perform" in question_lower or "sales" in question_lower):
        return """
[SALES REPORT]
   Product: Product Z
   Region: EMEIA
   Quarter: Q2_2025

   Revenue: $650,000
   Units Sold: 520
   Quarterly Growth: +30%
   YoY Growth: +34%
"""

    elif "forecast" in question_lower and "product z" in question_lower:
        return """
[FORECAST]
   Product: Product Z
   Current Value: $890,000
   Assumed Growth: 25%

   Forecast: $1,112,500
"""

    elif "alert" in question_lower or "send" in question_lower:
        return """
[ALERT PENDING]
   This action requires human approval.

   Please type 'approve' to send or 'reject' to cancel.
"""

    else:
        # Use LLM for other questions
        response = llm.invoke([{"role": "user", "content": question}])
        return response.content


# ============================================
# Agent Node
# ============================================

def agent_node(state: SalesAgentState):
    """Handle the user question."""

    print(f"\n[AGENT] Processing...")

    messages = state["messages"]
    last_message = messages[-1]
    question = last_message.get("content", "")

    # Get response
    response = handle_question(question)

    # Check if this is an alert that needs approval
    requires_approval = "[ALERT PENDING]" in response

    return {
        "messages": [{"role": "assistant", "content": response}],
        "requires_approval": requires_approval,
        "step": state.get("step", 0) + 1
    }


def human_approval_node(state: SalesAgentState):
    """Handle human approval for alerts."""

    print("\n" + "="*50)
    print("HUMAN APPROVAL REQUIRED")
    print("="*50)

    user_input = input("\nDo you approve sending this alert? (approve/reject): ").strip().lower()

    if user_input in ["approve", "yes", "y"]:
        return {
            "messages": [{"role": "assistant", "content": "[APPROVED] Alert has been sent to the team."}],
            "requires_approval": False,
            "step": state.get("step", 0) + 1
        }
    else:
        return {
            "messages": [{"role": "assistant", "content": "[REJECTED] Alert was cancelled."}],
            "requires_approval": False,
            "step": state.get("step", 0) + 1
        }


def router(state: SalesAgentState) -> Literal["human_approval", "end"]:
    """Decide next step."""

    if state.get("requires_approval", False):
        print("\n[ROUTER] -> HUMAN APPROVAL")
        return "human_approval"

    print("\n[ROUTER] -> END")
    return "end"


# ============================================
# Build Graph
# ============================================

workflow = StateGraph(SalesAgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("human_approval", human_approval_node)

workflow.set_entry_point("agent")

workflow.add_conditional_edges(
    "agent",
    router,
    {
        "human_approval": "human_approval",
        "end": END,
    }
)

workflow.add_edge("human_approval", END)

agent = workflow.compile()

print("\n[OK] Agent built successfully!")

[OK] LLM initialized

[OK] Agent built successfully!


In [53]:
# CELL 5: Test the agent

def ask_agent(question: str):
    """Send a question to the agent."""

    print("\n" + "="*60)
    print(f"USER: {question}")
    print("="*60)

    result = agent.invoke({
        "messages": [{"role": "user", "content": question}],
        "requires_approval": False,
        "step": 0
    })

    # Print response
    for msg in result["messages"]:
        if msg.get("role") == "assistant":
            print(f"\nASSISTANT: {msg['content']}")

    return result


# ============================================
# TEST 1: Sales data query
# ============================================

print("\n" + "="*30)
print("TEST 1: Sales data query")
print("="*30)

ask_agent("How did Product X perform in Q2 2025?")


# ============================================
# TEST 2: Forecast
# ============================================

print("\n" + "="*30)
print("TEST 2: Forecast calculation")
print("="*30)

ask_agent("What is the forecast for Product Z with 25% growth?")


# ============================================
# TEST 3: Alert with approval
# ============================================

print("\n" + "="*30)
print("TEST 3: Send alert (requires approval)")
print("="*30)

ask_agent("Send an alert to the UK Sales team about Q3 results")


TEST 1: Sales data query

USER: How did Product X perform in Q2 2025?

[AGENT] Processing...

[ROUTER] -> END

ASSISTANT: 
[SALES REPORT]
   Product: Product X
   Region: EMEIA
   Quarter: Q2_2025
   
   Revenue: $1,800,000
   Units Sold: 1,450
   Quarterly Growth: +20%
   YoY Growth: +20%


TEST 2: Forecast calculation

USER: What is the forecast for Product Z with 25% growth?

[AGENT] Processing...

[ROUTER] -> END

ASSISTANT: 
[FORECAST]
   Product: Product Z
   Current Value: $890,000
   Assumed Growth: 25%
   
   Forecast: $1,112,500


TEST 3: Send alert (requires approval)

USER: Send an alert to the UK Sales team about Q3 results

[AGENT] Processing...

[ROUTER] -> HUMAN APPROVAL

HUMAN APPROVAL REQUIRED

Do you approve sending this alert? (approve/reject): reject

ASSISTANT: [REJECTED] Alert was cancelled.


{'messages': [{'role': 'assistant',
   'content': '[REJECTED] Alert was cancelled.'}],
 'requires_approval': False,
 'step': 2}